In [15]:
import pandas as pd
import numpy as np
from datetime import datetime

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [16]:
# Load dataset
df = pd.read_csv("car data.csv")

# Show first 5 rows
print(df.head())

# Shape of dataset
print("Shape:", df.shape)

# Column names
print("Columns:", df.columns.tolist())

  Car_Name  Year  Selling_Price  Present_Price  Driven_kms Fuel_Type  \
0     ritz  2014           3.35           5.59       27000    Petrol   
1      sx4  2013           4.75           9.54       43000    Diesel   
2     ciaz  2017           7.25           9.85        6900    Petrol   
3  wagon r  2011           2.85           4.15        5200    Petrol   
4    swift  2014           4.60           6.87       42450    Diesel   

  Selling_type Transmission  Owner  
0       Dealer       Manual      0  
1       Dealer       Manual      0  
2       Dealer       Manual      0  
3       Dealer       Manual      0  
4       Dealer       Manual      0  
Shape: (301, 9)
Columns: ['Car_Name', 'Year', 'Selling_Price', 'Present_Price', 'Driven_kms', 'Fuel_Type', 'Selling_type', 'Transmission', 'Owner']


In [17]:
# Check data types
print(df.info())

# Check missing values
print(df.isnull().sum())

# Statistical summary
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Car_Name       301 non-null    object 
 1   Year           301 non-null    int64  
 2   Selling_Price  301 non-null    float64
 3   Present_Price  301 non-null    float64
 4   Driven_kms     301 non-null    int64  
 5   Fuel_Type      301 non-null    object 
 6   Selling_type   301 non-null    object 
 7   Transmission   301 non-null    object 
 8   Owner          301 non-null    int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 21.3+ KB
None
Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Driven_kms       0
Fuel_Type        0
Selling_type     0
Transmission     0
Owner            0
dtype: int64
              Year  Selling_Price  Present_Price     Driven_kms       Owner
count   301.000000     301.000000     301.000000     301.000000  301.000000
mean   2

In [18]:
current_year = datetime.now().year

df["Car_Age"] = current_year - df["Year"]

# Drop Year because Car_Age is more useful
df.drop("Year", axis=1, inplace=True)

print(df.head())

  Car_Name  Selling_Price  Present_Price  Driven_kms Fuel_Type Selling_type  \
0     ritz           3.35           5.59       27000    Petrol       Dealer   
1      sx4           4.75           9.54       43000    Diesel       Dealer   
2     ciaz           7.25           9.85        6900    Petrol       Dealer   
3  wagon r           2.85           4.15        5200    Petrol       Dealer   
4    swift           4.60           6.87       42450    Diesel       Dealer   

  Transmission  Owner  Car_Age  
0       Manual      0       12  
1       Manual      0       13  
2       Manual      0        9  
3       Manual      0       15  
4       Manual      0       12  


In [19]:
X = df.drop("Selling_Price", axis=1)
y = df["Selling_Price"]

print("Features:")
print(X.head())

print("Target:")
print(y.head())

Features:
  Car_Name  Present_Price  Driven_kms Fuel_Type Selling_type Transmission  \
0     ritz           5.59       27000    Petrol       Dealer       Manual   
1      sx4           9.54       43000    Diesel       Dealer       Manual   
2     ciaz           9.85        6900    Petrol       Dealer       Manual   
3  wagon r           4.15        5200    Petrol       Dealer       Manual   
4    swift           6.87       42450    Diesel       Dealer       Manual   

   Owner  Car_Age  
0      0       12  
1      0       13  
2      0        9  
3      0       15  
4      0       12  
Target:
0    3.35
1    4.75
2    7.25
3    2.85
4    4.60
Name: Selling_Price, dtype: float64


In [20]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

Categorical columns: ['Car_Name', 'Fuel_Type', 'Selling_type', 'Transmission']
Numerical columns: ['Present_Price', 'Driven_kms', 'Owner', 'Car_Age']


In [21]:
# Numerical preprocessing
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing
preprocessor = ColumnTransformer(transformers=[
    ("num", num_transformer, numerical_cols),
    ("cat", cat_transformer, categorical_cols)
])

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (240, 8)
X_test shape: (61, 8)


In [24]:
from sklearn.linear_model import LinearRegression

lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

print("Linear Regression")
print("MAE:", mean_absolute_error(y_test, lr_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, lr_pred)))
print("R2:", r2_score(y_test, lr_pred))

Linear Regression
MAE: 1.0323825461125498
RMSE: 1.5804075655760335
R2: 0.89157262012909


In [25]:
from sklearn.tree import DecisionTreeRegressor

dt_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(random_state=42))
])

dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

print("Decision Tree Regressor")
print("MAE:", mean_absolute_error(y_test, dt_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, dt_pred)))
print("R2:", r2_score(y_test, dt_pred))

Decision Tree Regressor
MAE: 0.6378688524590164
RMSE: 1.0286478471481981
R2: 0.9540660063342169


In [26]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=200, random_state=42))
])

rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Random Forest Regressor")
print("MAE:", mean_absolute_error(y_test, rf_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, rf_pred)))
print("R2:", r2_score(y_test, rf_pred))

Random Forest Regressor
MAE: 0.6272860655737699
RMSE: 0.9498651214216903
R2: 0.9608326088665678


In [29]:
# -------------------------
# Model Comparison
# -------------------------

import pandas as pd

results = pd.DataFrame({
    "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
    
    "MAE": [
        mean_absolute_error(y_test, lr_pred),
        mean_absolute_error(y_test, dt_pred),
        mean_absolute_error(y_test, rf_pred)
    ],
    
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, lr_pred)),
        np.sqrt(mean_squared_error(y_test, dt_pred)),
        np.sqrt(mean_squared_error(y_test, rf_pred))
    ],
    
    "R2 Score": [
        r2_score(y_test, lr_pred),
        r2_score(y_test, dt_pred),
        r2_score(y_test, rf_pred)
    ]
})

# Sort by best model (highest R2)
results = results.sort_values(by="R2 Score", ascending=False)

print("\nFinal Model Comparison:\n")
print(results)


Final Model Comparison:

               Model       MAE      RMSE  R2 Score
2      Random Forest  0.627286  0.949865  0.960833
1      Decision Tree  0.637869  1.028648  0.954066
0  Linear Regression  1.032383  1.580408  0.891573


In [31]:
best_model_name = results.iloc[0]["Model"]
print("\nBest Model is:", best_model_name)


Best Model is: Random Forest


In [37]:
# -------------------------
# Find best model
# -------------------------
best_model_name = results.iloc[0]["Model"]

if best_model_name == "Linear Regression":
    best_model = lr_model
elif best_model_name == "Decision Tree":
    best_model = dt_model
else:
    best_model = rf_model

print("\nBest Model is:", best_model_name)

# -------------------------
# Predict on new data
# -------------------------
new_data = pd.DataFrame({
    "Car_Name": ["ritz"],
    "Present_Price": [5.59],
    "Driven_kms": [27000],
    "Fuel_Type": ["Petrol"],
    "Selling_type": ["Dealer"],
    "Transmission": ["Manual"],
    "Owner": [0],
    "Car_Age": [datetime.now().year - 2014]
})

prediction = best_model.predict(new_data)

print("Predicted Selling Price:", prediction[0])


Best Model is: Random Forest
Predicted Selling Price: 3.789500000000008


In [38]:
import joblib

# (your full training + comparison code here)

# Save best model
if best_model_name == "Linear Regression":
    best_model = lr_model
elif best_model_name == "Decision Tree":
    best_model = dt_model
else:
    best_model = rf_model

joblib.dump(best_model, "car_price_model.pkl")

print("Best model saved as car_model.pkl")

Best model saved as car_model.pkl
